# Project name: Demand-Hype Divergence Score Prediction

## 📝 Project info:

#### 🏭 Industry  
- E-commerce & Marketing 

#### 🌐 Domain  
- Lifestyle & Consumer Themes  

## 📂 Dataset Information

#### 🔎 Sources
- **GDELT GKG** (Global Knowledge Graph)  
- **Google Trends**  
- **World Bank Open Data (WDI)**
- **Holidays(Python Library)**
#### All the datasets are **open source** and publicly available.  
- ✅ No privacy, licensing, or access restrictions  
- ✅ Safe for research and personal use 

#### Problem Name: Demand-Hype Divergence Score Prediction

#### ML Task: Regression

##### Problem Statement:
A key strategic question: is a market being over-covered by media relative to actual consumer demand, or is there an untapped demand opportunity? This model predicts a "Divergence Score" — a derived continuous target — that quantifies how much a market deviates from the expected demand-to-coverage balance, enabling media budget reallocation and editorial prioritization.

##### Why This Problem:
EDA showed Demand_to_Hype_Ratio ↔ Net_Sentiment | r = 0.316 and ↔ Activity_Density | r = -0.287 — sentiment and media activity partially explain demand-media divergence. This regression estimates why divergence happens and gives brands a forward-looking signal.

##### Expected ML Output:
A continuous Divergence_Score per country-category-week. Positive values flag under-served opportunities; negative values flag over-hyped markets.


### 💻 Code Work:

In [3]:
import pandas as pd
import numpy as np
import mysql.connector
import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

### 📤 1. Data Loading (Loading Datafrom Database tables into pandas dataframe df)

In [4]:
# connecting to the MySQL database
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="!@Mstrong@1234",
    database="GDELTTrends"
)

In [5]:
# initailizing the cursor
cursor = connection.cursor()
# getting table names from the database
cursor.execute("SHOW TABLES")
# fetching the table names
tables = cursor.fetchall()
print("Tables in GDELTTrends:")
for table in cursor.fetchall():
    print(table[0])

Tables in GDELTTrends:


In [6]:
df = pd.read_sql("SELECT * FROM final_master_dataset;", connection)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8022 entries, 0 to 8021
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Week_Start               8022 non-null   datetime64[ns]
 1   Country_Code             8022 non-null   object        
 2   Country_Name             8022 non-null   object        
 3   Category                 8022 non-null   object        
 4   Search_Interest          8022 non-null   float64       
 5   Search_Velocity          8022 non-null   float64       
 6   Search_Acceleration      8022 non-null   float64       
 7   Media_Volume             7725 non-null   float64       
 8   tone_net_sentiment       7725 non-null   float64       
 9   tone_positive_score      7725 non-null   float64       
 10  tone_negative_score      7725 non-null   float64       
 11  tone_polarity            7725 non-null   float64       
 12  tone_activity_density    7725 non-

### 👩🏻‍💻 1. Modeling

#### 🛠️ 1.1 Pre-Requisites

### 🎯 1.1.2 X & y

#### Features (X-columns)

| Feature | Role |
| --- | --- |
| Week_Start | Seasonality (time reference) |
| Country_Name | Segment context (categorical) |
| Category | Segment context (categorical) |
| Search_Velocity | Demand momentum (rate of change) |
| Search_Acceleration | Demand momentum (second derivative) |
| Media_Volume | Coverage intensity |
| tone_net_sentiment | Sentiment context |
| tone_polarity | Emotional tone of coverage |
| tone_activity_density | Event-driven coverage intensity |
| Source_Diversity | Coverage breadth |
| Inflation_Rate | Economic demand amplifier |
| Internet_Penetration | Media ecosystem capacity |
| holiday_count | Calendar demand spikes (numeric) |

---

#### Target (y-column)

| Target | Description |
| --- | --- |
| Divergence_Score | Regression target |

---

#### X-columns that need to be created (Feature Engineering)

| New Feature | Role |
| --- | --- |
| Search_Velocity_Lag1, Search_Velocity_Lag2 | Lagged demand momentum |
| Quarter | Seasonality (derived from Week_Start) |
| Month | Seasonality (derived from Week_Start) |

### Columns to Exclude

| Column | Reason for Exclusion |
| --- | --- |
| Country_Code | Redundant with Country_Name (categorical already included) |
| Search_Interest | Raw demand signal, replaced by velocity/acceleration features |
| tone_positive_score | Subsumed by tone_net_sentiment & tone_polarity |
| tone_negative_score | Subsumed by tone_net_sentiment & tone_polarity |
| tone_self_group_density | Not directly relevant to divergence modeling |
| Demand_to_Hype_Ratio | Derived metric, excluded to avoid leakage |
| Media_Volume_per_Source | Redundant with Media_Volume & Source_Diversity |
| Interest_per_Source | Redundant with Search_Velocity & Search_Acceleration |
| GDP_Per_Capita | Economic context, excluded to reduce collinearity |
| Year | Seasonality captured via Month/Quarter/Week_Start instead |
| is_holiday | Binary flag less informative than holiday_count |


#### ✂️ 1.1.2 Train-Test Split:

In [8]:
df.columns

Index(['Week_Start', 'Country_Code', 'Country_Name', 'Category',
       'Search_Interest', 'Search_Velocity', 'Search_Acceleration',
       'Media_Volume', 'tone_net_sentiment', 'tone_positive_score',
       'tone_negative_score', 'tone_polarity', 'tone_activity_density',
       'tone_self_group_density', 'Source_Diversity', 'Demand_to_Hype_Ratio',
       'Media_Volume_per_Source', 'Interest_per_Source', 'Year',
       'Inflation_Rate', 'Internet_Penetration', 'GDP_Per_Capita',
       'is_holiday', 'holiday_count'],
      dtype='object')

In [9]:
# splitting horizontally
train_df = df[(df['Week_Start'].dt.year >= 2023) & (df['Week_Start'].dt.year <= 2025)]
test_df = df[(df['Week_Start'] >= '2026-02-01') & (df['Week_Start'] < '2026-08-01')]

In [10]:
print(f"length of X_test {len(train_df)}\nlength of X_train {len(test_df)}")

length of X_test 6594
length of X_train 1092


In [11]:
# Deriving y column from the dataframe
# Divergence_Score = Search_Interest_norm - log_Media_Volume_norm
# log_Media_Volume = log1p(Media_Volume)
# Search_Interest_norm = (Search_Interest - Min_Category) / (Max_Category - Min_Category)
# log_Media_Volume_norm = (log_Media_Volume - Min_Category) / (Max_Category - Min_Category)

# since Media_Volume is having null values, we fill those null values with 0 and then procede with the calculation of log_Media_Volume and log_Media_Volume_norm
# Fill missing Media_Volume values with 0 for both sets
train_df['Media_Volume'] = train_df['Media_Volume'].fillna(0)
test_df['Media_Volume'] = test_df['Media_Volume'].fillna(0)

# Compute log_Media_Volume
train_df['log_Media_Volume'] = np.log1p(train_df['Media_Volume'])
test_df['log_Media_Volume'] = np.log1p(test_df['Media_Volume'])

# Calculate min and max strictly from the TRAINING set (prevents data leakage)
si_min = train_df['Search_Interest'].min()
si_max = train_df['Search_Interest'].max()

log_mv_min = train_df['log_Media_Volume'].min()
log_mv_max = train_df['log_Media_Volume'].max()

# Normalize Search_Interest using training stats
train_si_norm = (train_df['Search_Interest'] - si_min) / (si_max - si_min)
test_si_norm = (test_df['Search_Interest'] - si_min) / (si_max - si_min)

# Normalize log_Media_Volume using training stats
train_log_mv_norm = (train_df['log_Media_Volume'] - log_mv_min) / (log_mv_max - log_mv_min)
test_log_mv_norm = (test_df['log_Media_Volume'] - log_mv_min) / (log_mv_max - log_mv_min)

# Derive Divergence_Score for both sets
train_df['Divergence_Score'] = train_si_norm - train_log_mv_norm
test_df['Divergence_Score'] = test_si_norm - test_log_mv_norm

In [12]:
# Separate features and target for training
y_train = train_df['Divergence_Score']
X_train = train_df.drop(columns=['Divergence_Score','log_Media_Volume'])

# Separate features and target for testing
y_test = test_df['Divergence_Score']
X_test = test_df.drop(columns=['Divergence_Score','log_Media_Volume'])

#### 🔄 1.1.3 Missing Values & Outliers Handling 

In [13]:
null_cols_X_train = X_train.columns[X_train.isnull().any()]

In [14]:
null_cols_X_test = X_test.columns[X_test.isnull().any()]

In [15]:
# null Values in all the columns can be replaced with 0, since they are real world values is 0, it is a valid value for them.
for col in null_cols_X_train:
    X_train[col] = X_train[col].fillna(0)

In [16]:
# null Values in all the columns can be replaced with 0, since they are real world values is 0, it is a valid value for them.
for col in null_cols_X_test:
    X_test[col] = X_test[col].fillna(0)

In [17]:
X_train.isnull().sum()

Week_Start                 0
Country_Code               0
Country_Name               0
Category                   0
Search_Interest            0
Search_Velocity            0
Search_Acceleration        0
Media_Volume               0
tone_net_sentiment         0
tone_positive_score        0
tone_negative_score        0
tone_polarity              0
tone_activity_density      0
tone_self_group_density    0
Source_Diversity           0
Demand_to_Hype_Ratio       0
Media_Volume_per_Source    0
Interest_per_Source        0
Year                       0
Inflation_Rate             0
Internet_Penetration       0
GDP_Per_Capita             0
is_holiday                 0
holiday_count              0
dtype: int64

In [18]:
X_test.isnull().sum()

Week_Start                 0
Country_Code               0
Country_Name               0
Category                   0
Search_Interest            0
Search_Velocity            0
Search_Acceleration        0
Media_Volume               0
tone_net_sentiment         0
tone_positive_score        0
tone_negative_score        0
tone_polarity              0
tone_activity_density      0
tone_self_group_density    0
Source_Diversity           0
Demand_to_Hype_Ratio       0
Media_Volume_per_Source    0
Interest_per_Source        0
Year                       0
Inflation_Rate             0
Internet_Penetration       0
GDP_Per_Capita             0
is_holiday                 0
holiday_count              0
dtype: int64

In [19]:
print(y_test.isnull().sum())
print(y_train.isnull().sum())

0
0


##### Missing values are handled, no null values in x columns
##### for this real world dataset, there is no need for handling outliers, based on model output I'll look into this aspect

#### ⚙️ 1.1.4 Feature Engineering 
####  **Feature Generation** 
##### As mentioned above we need to create features

- Search_Velocity_Lag1
- Search_Velocity_Lag2
- Quarter
- Month


#### Calculating Search_Velocity_Lag1, Search_Velocity_Lag2

In [20]:
## in order not to loose the test data due to split that is performed before as X_test and X_train
## Performing Calculation on main dataframe df
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8022 entries, 0 to 8021
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Week_Start               8022 non-null   datetime64[ns]
 1   Country_Code             8022 non-null   object        
 2   Country_Name             8022 non-null   object        
 3   Category                 8022 non-null   object        
 4   Search_Interest          8022 non-null   float64       
 5   Search_Velocity          8022 non-null   float64       
 6   Search_Acceleration      8022 non-null   float64       
 7   Media_Volume             7725 non-null   float64       
 8   tone_net_sentiment       7725 non-null   float64       
 9   tone_positive_score      7725 non-null   float64       
 10  tone_negative_score      7725 non-null   float64       
 11  tone_polarity            7725 non-null   float64       
 12  tone_activity_density    7725 non-

In [21]:
# calculating lag features
df['Search_Velocity_Lag1'] = df.groupby(['Country_Name','Category'])['Search_Velocity'].shift(1)
df['Search_Velocity_Lag2'] = df.groupby(['Country_Name','Category'])['Search_Velocity'].shift(2)

In [22]:
# merging the calculated features back to X_train
train_xdf = df[(df['Week_Start'].dt.year >= 2023) & (df['Week_Start'].dt.year <= 2025)]
X_train['Search_Velocity_Lag1'] = train_xdf['Search_Velocity_Lag1']
X_train['Search_Velocity_Lag2'] = train_xdf['Search_Velocity_Lag2']

In [23]:
# merging the calculated features back to X_test
test_xdf = df[(df['Week_Start'] >= '2026-02-01') & (df['Week_Start'] < '2026-08-01')]
X_test['Search_Velocity_Lag1'] = test_xdf['Search_Velocity_Lag1']
X_test['Search_Velocity_Lag2'] = test_xdf['Search_Velocity_Lag2']

In [24]:
print("Null values in X_train",X_train.isnull().sum())

Null values in X_train Week_Start                  0
Country_Code                0
Country_Name                0
Category                    0
Search_Interest             0
Search_Velocity             0
Search_Acceleration         0
Media_Volume                0
tone_net_sentiment          0
tone_positive_score         0
tone_negative_score         0
tone_polarity               0
tone_activity_density       0
tone_self_group_density     0
Source_Diversity            0
Demand_to_Hype_Ratio        0
Media_Volume_per_Source     0
Interest_per_Source         0
Year                        0
Inflation_Rate              0
Internet_Penetration        0
GDP_Per_Capita              0
is_holiday                  0
holiday_count               0
Search_Velocity_Lag1       42
Search_Velocity_Lag2       84
dtype: int64


In [25]:
print("Null values in X_test", X_test.isnull().sum())

Null values in X_test Week_Start                 0
Country_Code               0
Country_Name               0
Category                   0
Search_Interest            0
Search_Velocity            0
Search_Acceleration        0
Media_Volume               0
tone_net_sentiment         0
tone_positive_score        0
tone_negative_score        0
tone_polarity              0
tone_activity_density      0
tone_self_group_density    0
Source_Diversity           0
Demand_to_Hype_Ratio       0
Media_Volume_per_Source    0
Interest_per_Source        0
Year                       0
Inflation_Rate             0
Internet_Penetration       0
GDP_Per_Capita             0
is_holiday                 0
holiday_count              0
Search_Velocity_Lag1       0
Search_Velocity_Lag2       0
dtype: int64


### lag features are calulated for with past 2 weeks of search velocity as part of handling null values excluding 2 weeks of data 

In [26]:
set(X_train['Week_Start'][X_train['Search_Velocity_Lag2'].isnull()].to_list())

{Timestamp('2023-01-01 00:00:00'), Timestamp('2023-01-08 00:00:00')}

In [27]:
set(X_test['Week_Start'][X_test['Search_Velocity_Lag2'].isnull()].to_list())

set()

In [28]:
# Excluding 2 weeks data from X_train
X_train = X_train[X_train['Week_Start'] > '2023-01-08']

In [29]:
# Excluding 2 weeds data from y_train
y_train_df = train_df[train_df['Week_Start'] > '2023-01-08']
y_train = y_train_df['Divergence_Score']

In [30]:
len(X_train) == len(y_train)

True

In [31]:
X_train.isnull().sum()

Week_Start                 0
Country_Code               0
Country_Name               0
Category                   0
Search_Interest            0
Search_Velocity            0
Search_Acceleration        0
Media_Volume               0
tone_net_sentiment         0
tone_positive_score        0
tone_negative_score        0
tone_polarity              0
tone_activity_density      0
tone_self_group_density    0
Source_Diversity           0
Demand_to_Hype_Ratio       0
Media_Volume_per_Source    0
Interest_per_Source        0
Year                       0
Inflation_Rate             0
Internet_Penetration       0
GDP_Per_Capita             0
is_holiday                 0
holiday_count              0
Search_Velocity_Lag1       0
Search_Velocity_Lag2       0
dtype: int64

### Created 2 feautes
- Search_Velocity_Lag1
- Search_Velocity_Lag2
### null values also handled

In [32]:
X_train.columns

Index(['Week_Start', 'Country_Code', 'Country_Name', 'Category',
       'Search_Interest', 'Search_Velocity', 'Search_Acceleration',
       'Media_Volume', 'tone_net_sentiment', 'tone_positive_score',
       'tone_negative_score', 'tone_polarity', 'tone_activity_density',
       'tone_self_group_density', 'Source_Diversity', 'Demand_to_Hype_Ratio',
       'Media_Volume_per_Source', 'Interest_per_Source', 'Year',
       'Inflation_Rate', 'Internet_Penetration', 'GDP_Per_Capita',
       'is_holiday', 'holiday_count', 'Search_Velocity_Lag1',
       'Search_Velocity_Lag2'],
      dtype='object')

### Proceding to next step without creating columns for month and year
### Based on model performance decide further to create the columns

**Feature selection**

In [33]:
X_train=X_train.drop(["Country_Code","Search_Interest","tone_positive_score","tone_negative_score","tone_self_group_density","Demand_to_Hype_Ratio","Media_Volume_per_Source","Interest_per_Source","GDP_Per_Capita","Year","is_holiday"],axis=1)
X_test= X_test.drop(["Country_Code","Search_Interest","tone_positive_score","tone_negative_score","tone_self_group_density","Demand_to_Hype_Ratio","Media_Volume_per_Source","Interest_per_Source","GDP_Per_Capita","Year","is_holiday"],axis=1)

In [34]:
print("X_train Columns",X_train.columns)
print("X_test Columns", X_test.columns)

X_train Columns Index(['Week_Start', 'Country_Name', 'Category', 'Search_Velocity',
       'Search_Acceleration', 'Media_Volume', 'tone_net_sentiment',
       'tone_polarity', 'tone_activity_density', 'Source_Diversity',
       'Inflation_Rate', 'Internet_Penetration', 'holiday_count',
       'Search_Velocity_Lag1', 'Search_Velocity_Lag2'],
      dtype='object')
X_test Columns Index(['Week_Start', 'Country_Name', 'Category', 'Search_Velocity',
       'Search_Acceleration', 'Media_Volume', 'tone_net_sentiment',
       'tone_polarity', 'tone_activity_density', 'Source_Diversity',
       'Inflation_Rate', 'Internet_Penetration', 'holiday_count',
       'Search_Velocity_Lag1', 'Search_Velocity_Lag2'],
      dtype='object')


#### Removed unnecessary columns from x_columns

**Feature Modification:**

**Encoding **

In [35]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1092 entries, 483 to 8009
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Week_Start             1092 non-null   datetime64[ns]
 1   Country_Name           1092 non-null   object        
 2   Category               1092 non-null   object        
 3   Search_Velocity        1092 non-null   float64       
 4   Search_Acceleration    1092 non-null   float64       
 5   Media_Volume           1092 non-null   float64       
 6   tone_net_sentiment     1092 non-null   float64       
 7   tone_polarity          1092 non-null   float64       
 8   tone_activity_density  1092 non-null   float64       
 9   Source_Diversity       1092 non-null   float64       
 10  Inflation_Rate         1092 non-null   float64       
 11  Internet_Penetration   1092 non-null   float64       
 12  holiday_count          1092 non-null   int64         
 13  Search

#### Categorical columns in dataset - Country_Name, Category - both are nominal columns

In [36]:
# Identify categorical columns from X_train
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

# Initialize OneHotEncoder 
# handle_unknown='ignore' prevents errors on new categories in X_test
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on X_train categories and transform both X_train and X_test
train_encoded = encoder.fit_transform(X_train[categorical_cols])
test_encoded = encoder.transform(X_test[categorical_cols])

# Generate column names for the encoded features
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

# Convert encoded arrays back to DataFrames with original indices
train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_feature_names, index=X_train.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_feature_names, index=X_test.index)

# Drop original categorical columns and join the new encoded columns
X_train = X_train.drop(columns=categorical_cols).join(train_encoded_df)
X_test = X_test.drop(columns=categorical_cols).join(test_encoded_df)

In [37]:
X_train.describe()

,Week_Start,Search_Velocity,Search_Acceleration,Media_Volume,tone_net_sentiment,tone_polarity,tone_activity_density,Source_Diversity,Inflation_Rate,Internet_Penetration,...,Country_Name_Mexico,Country_Name_Nigeria,Country_Name_Singapore,Country_Name_South_Africa,Country_Name_United_Arab_Emirates,Country_Name_United_Kingdom,Country_Name_United_States,Category_Fashion_Beauty,Category_Fitness_Wearables,Category_Nutrition_Diets
count,6510,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,...,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000,6510.000000
mean,2024-07-07 00:00:00,0.030327,0.026986,4905.445315,-0.205679,6.760165,20.145840,807.511367,5.579180,81.255065,...,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.071429,0.333333,0.333333,0.333333
min,2023-01-15 00:00:00,-25.340000,-50.140000,0.000000,-9.980800,0.000000,0.000000,0.000000,0.220000,32.070000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2023-10-08 00:00:00,-1.930000,-2.777500,37.000000,-1.387367,6.298293,19.801600,25.000000,2.390000,78.360000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2024-07-07 00:00:00,-0.110000,0.130000,241.000000,-0.204272,7.015045,20.941850,134.500000,4.360000,89.625000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2025-04-06 00:00:00,1.847500,2.980000,1779.000000,0.674002,7.635885,22.109600,629.500000,4.950000,94.690000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
max,2025-12-28 00:00:00,32.980000,30.430000,168862.000000,7.331040,13.924100,30.830600,12317.000000,33.240000,100.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,3.984991,5.861034,18527.536323,1.639797,1.700916,4.491891,1789.641084,7.169653,20.092086,...,0.257559,0.257559,0.257559,0.257559,0.257559,0.257559,0.257559,0.471441,0.471441,0.471441


In [38]:
# Drop the non-numeric 'Week_Start' column from X_train and X_test before modeling
if 'Week_Start' in X_train.columns:
    X_train = X_train.drop(columns=['Week_Start'])
if 'Week_Start' in X_test.columns:
    X_test = X_test.drop(columns=['Week_Start'])

In [39]:
X_train.select_dtypes(include=['int64','float64']).columns

Index(['Search_Velocity', 'Search_Acceleration', 'Media_Volume',
       'tone_net_sentiment', 'tone_polarity', 'tone_activity_density',
       'Source_Diversity', 'Inflation_Rate', 'Internet_Penetration',
       'holiday_count', 'Search_Velocity_Lag1', 'Search_Velocity_Lag2',
       'Country_Name_Australia', 'Country_Name_Brazil', 'Country_Name_Canada',
       'Country_Name_China', 'Country_Name_France', 'Country_Name_India',
       'Country_Name_Kenya', 'Country_Name_Mexico', 'Country_Name_Nigeria',
       'Country_Name_Singapore', 'Country_Name_South_Africa',
       'Country_Name_United_Arab_Emirates', 'Country_Name_United_Kingdom',
       'Country_Name_United_States', 'Category_Fashion_Beauty',
       'Category_Fitness_Wearables', 'Category_Nutrition_Diets'],
      dtype='object')

In [40]:
# Drop non-numeric/identifier columns that shouldn't be scaled (like Week_Start)
cols_to_scale = [
    "Search_Velocity",
    "Search_Acceleration",
    "Media_Volume",
    "tone_net_sentiment",
    "tone_polarity",
    "tone_activity_density",
    "Source_Diversity",
    "Inflation_Rate",
    "Internet_Penetration",
    "holiday_count",
    "Search_Velocity_Lag1",
    "Search_Velocity_Lag2"
]

# Check whether all required columns exist
missing_cols = [col for col in cols_to_scale if col not in X_train.columns]

print("Missing columns:", missing_cols)

# Keep only columns actually available in BOTH train and test
cols_to_scale = [
    col for col in cols_to_scale
    if col in X_train.columns and col in X_test.columns
]

# Make explicit copies to avoid SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

# Initialize scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train.loc[:, cols_to_scale] = scaler.fit_transform(
    X_train[cols_to_scale]
)

# Use the already-fitted scaler on test data
X_test.loc[:, cols_to_scale] = scaler.transform(
    X_test[cols_to_scale]
)

Missing columns: []


In [41]:
X_test.isnull().sum()

Search_Velocity                      0
Search_Acceleration                  0
Media_Volume                         0
tone_net_sentiment                   0
tone_polarity                        0
tone_activity_density                0
Source_Diversity                     0
Inflation_Rate                       0
Internet_Penetration                 0
holiday_count                        0
Search_Velocity_Lag1                 0
Search_Velocity_Lag2                 0
Country_Name_Australia               0
Country_Name_Brazil                  0
Country_Name_Canada                  0
Country_Name_China                   0
Country_Name_France                  0
Country_Name_India                   0
Country_Name_Kenya                   0
Country_Name_Mexico                  0
Country_Name_Nigeria                 0
Country_Name_Singapore               0
Country_Name_South_Africa            0
Country_Name_United_Arab_Emirates    0
Country_Name_United_Kingdom          0
Country_Name_United_State

#### 🤖 1.2 Model + Define, Train & Study

In [42]:
X_train.columns

Index(['Search_Velocity', 'Search_Acceleration', 'Media_Volume',
       'tone_net_sentiment', 'tone_polarity', 'tone_activity_density',
       'Source_Diversity', 'Inflation_Rate', 'Internet_Penetration',
       'holiday_count', 'Search_Velocity_Lag1', 'Search_Velocity_Lag2',
       'Country_Name_Australia', 'Country_Name_Brazil', 'Country_Name_Canada',
       'Country_Name_China', 'Country_Name_France', 'Country_Name_India',
       'Country_Name_Kenya', 'Country_Name_Mexico', 'Country_Name_Nigeria',
       'Country_Name_Singapore', 'Country_Name_South_Africa',
       'Country_Name_United_Arab_Emirates', 'Country_Name_United_Kingdom',
       'Country_Name_United_States', 'Category_Fashion_Beauty',
       'Category_Fitness_Wearables', 'Category_Nutrition_Diets'],
      dtype='object')

**Regression:**

In [43]:
# Define all regression models
models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression (Degree=2)": Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('linear', LinearRegression())
    ]),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel='rbf'),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, learning_rate=0.1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Heuristic to determine fit status
    if train_r2 < 0.1 and test_r2 < 0.1:
        fit_status = "Underfit"
    elif (train_r2 - test_r2) > 0.15:
        fit_status = "Overfit"
    else:
        fit_status = "Goodfit"
        
    results.append({
        "Models": name,
        "TrainLoss (RMSE)": round(train_rmse, 2),
        "TestLoss (RMSE)": round(test_rmse, 2),
        "TrainScore (R²)": round(train_r2, 2),
        "TestScore (R²)": round(test_r2, 2),
        "Bias-Variance (Fit)": fit_status
    })

results_df = pd.DataFrame(results)

In [44]:
print("Final Evalaution Table")
display(results_df)

Final Evalaution Table


,Models,TrainLoss (RMSE),TestLoss (RMSE),TrainScore (R²),TestScore (R²),Bias-Variance (Fit)
0,Linear Regression,0.11,0.18,0.87,0.58,Overfit
1,Polynomial Regression (Degree=2),0.06,0.15,0.97,0.72,Overfit
2,Ridge,0.11,0.18,0.87,0.58,Overfit
3,Lasso,0.17,0.20,0.70,0.49,Overfit
4,KNN Regressor,0.07,0.17,0.95,0.61,Overfit
5,SVR,0.06,0.14,0.96,0.73,Overfit
6,Decision Tree,0.00,0.17,1.00,0.63,Overfit
7,Random Forest,0.02,0.15,1.00,0.71,Overfit
8,XGBoost,0.03,0.14,0.99,0.73,Overfit


In [45]:
#### ✨ 1.4 Hyp Param Tuning
##### As XGBoost is perming better let's proceed with that

In [46]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor


# Define the hyperparameter search space
param_distributions = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6,7,8],
    'min_child_weight': [3, 5, 10, 15],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'reg_alpha': [0.0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 3.0, 5.0]
}

# Initialize base model
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)

# Set up RandomizedSearchCV to explore the parameter space efficiently
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=30,             # Number of parameter combinations to try
    scoring='r2',
    cv=5,                  # 5-fold cross-validation on the training set
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit the random search to the training data
random_search.fit(X_train, y_train)

# Extract the best model
xgb = random_search.best_estimator_

print("Best Parameters Found:")
for param, val in random_search.best_params_.items():
    print(f"  {param}: {val}")

# Evaluate best model on train and test sets
y_train_pred = xgb.predict(X_train)
y_test_pred = xgb.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

if train_r2 < 0.1 and test_r2 < 0.1:
    fit_status = "Underfit"
elif (train_r2 - test_r2) > 0.15:  
    fit_status = "Overfit"
else:
    fit_status = "Goodfit"

optimized_result = {
    "Models": "XGBoost (RandomSearch Optimized)",
    "TrainLoss (RMSE)": round(train_rmse, 2),
    "TestLoss (RMSE)": round(test_rmse, 2),
    "TrainScore (R²)": round(train_r2, 2),
    "TestScore (R²)": round(test_r2, 2),
    "Bias-Variance (Fit)": fit_status
}

print("\nFinal Evaluation:")
print(pd.DataFrame([optimized_result]).to_string(index=False))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters Found:
  subsample: 0.9
  reg_lambda: 1.0
  reg_alpha: 0.1
  n_estimators: 300
  min_child_weight: 10
  max_depth: 6
  learning_rate: 0.05
  colsample_bytree: 0.6

Final Evaluation:
                          Models  TrainLoss (RMSE)  TestLoss (RMSE)  TrainScore (R²)  TestScore (R²) Bias-Variance (Fit)
XGBoost (RandomSearch Optimized)              0.04             0.14             0.99            0.76             Overfit


In [47]:
# Finalised XGBoost model and building model
xgb_tuned = XGBRegressor(
    n_estimators=300,            # More trees with a lower learning rate for stable generalization
    learning_rate=0.05,          # Slower learning rate prevents rapid memorization of training data
    max_depth=6,                 # Shallow trees prevent complex local decision boundaries
    min_child_weight=10,          # Requires more samples per leaf, stopping micro-splits
    subsample=0.9,               # Samples 90% of rows per tree to reduce variance
    colsample_bytree=0.8,        # Samples 80% of features per tree to break feature dominance
    reg_alpha=0.1,               # L1 regularization (Lasso penalty on weights)
    reg_lambda=0.1,              # L2 regularization (Ridge penalty on weights)
    random_state=42,
    n_jobs=-1
)

# Fit on training data
xgb_tuned.fit(X_train, y_train)

# Predict on train and test sets
y_train_pred = xgb_tuned.predict(X_train)
y_test_pred = xgb_tuned.predict(X_test)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Determine fit status
if train_r2 < 0.1 and test_r2 < 0.1:
    fit_status = "Underfit"
elif (train_r2 - test_r2) > 0.15:
    fit_status = "Overfit"
else:
    fit_status = "Goodfit"

# Output final evaluation dictionary
tuned_result = {
    "Models": "XGBoost (Tuned)",
    "TrainLoss (RMSE)": round(train_rmse, 2),
    "TestLoss (RMSE)": round(test_rmse, 2),
    "TrainScore (R²)": round(train_r2, 2),
    "TestScore (R²)": round(test_r2, 2),
    "Bias-Variance (Fit)": fit_status
}

print(pd.DataFrame([tuned_result]).to_string(index=False))

         Models  TrainLoss (RMSE)  TestLoss (RMSE)  TrainScore (R²)  TestScore (R²) Bias-Variance (Fit)
XGBoost (Tuned)              0.03             0.14             0.99            0.75             Overfit


#### 📥 1.5 Saving model & Real Time Prediction

In [48]:
# Define your target folder path
folder_path = r"E:\\ML Project Internship\\ML-Project\\app\\pickles"
os.makedirs(folder_path, exist_ok=True)

# File paths
model_filename   = os.path.join(folder_path, "xgb_divergence_model.pkl")
encoder_filename = os.path.join(folder_path, "onehot_encoder.pkl")
scaler_filename  = os.path.join(folder_path, "scaler.pkl")

try:
    # Save model
    joblib.dump(xgb_tuned, model_filename)
    print(f"✅ Model successfully saved to {model_filename}")
except Exception as e:
    print(f"❌ Error saving model: {e}")

try:
    # Save encoder
    joblib.dump(encoder, encoder_filename)
    print(f"✅ Encoder successfully saved to {encoder_filename}")
except Exception as e:
    print(f"❌ Error saving encoder: {e}")

try:
    # Save scaler
    joblib.dump(scaler, scaler_filename)
    print(f"✅ Scaler successfully saved to {scaler_filename}")
except Exception as e:
    print(f"❌ Error saving scaler: {e}")


✅ Model successfully saved to E:\\ML Project Internship\\ML-Project\\app\\pickles\xgb_divergence_model.pkl
✅ Encoder successfully saved to E:\\ML Project Internship\\ML-Project\\app\\pickles\onehot_encoder.pkl
✅ Scaler successfully saved to E:\\ML Project Internship\\ML-Project\\app\\pickles\scaler.pkl


## 📂 Saved Model and Preprocessing Steps (App Folder)

All essential components for deployment have been saved in the `app/pickles` directory:

| Component        | File Name                  | Purpose                                      |
|------------------|----------------------------|----------------------------------------------|
| XGBoost Model    | `xgb_divergence_model.pkl` | Final tuned model for predictions             |
| OneHot Encoder   | `onehot_encoder.pkl`       | Handles categorical feature transformation    |
| Standard Scaler  | `scaler.pkl`               | Normalizes numeric features for consistency   |

✅ These files ensure that both the **model** and the **preprocessing pipeline** can be reloaded seamlessly for inference.
